[English](write_your_method.en.ipynb) | **한국어**

# Python Method 만들기

Python 함수를 Method로 등록하면 Decision Layer가 semantic layer에서 데이터를 가져와 함수에 넣어 실행합니다. 

AI는 필요한 지표와 기간을 찾고, 모호한 입력은 사용자에게 확인합니다. 서버에서 실행한 입력과 결과는 Run에 기록됩니다.

이 노트북에서는 **함수 작성 → 등록 → 입력 선택 → 결과 확인**을 순서대로 해봅니다.

1. 간단한 Python 함수를 Method로 등록하고 실행합니다.
2. 이어서 복잡한 분석인 OLS를 예로 여러 입력이 필요한 Method를 만들고 실행합니다.

이 노트북은 준비된 예제 데이터로 실행하므로 Cube 서버나 AI 연결이 필요하지 않습니다. 프로젝트에 저장해 Web과 AI에서 사용하는 방법은 [서비스 사용 가이드](../../ko/guides/method-projects.md)에서 이어집니다.

In [1]:
import sys
from pathlib import Path

import pandas as pd

# examples/ is outside the installed decision_layer package.
repo_root = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "examples/methods").is_dir()
)
sys.path.insert(0, str(repo_root))

from decision_layer.methods import Column, DataInput, Scope, python_method
from importlib import reload
from examples.methods.authoring import customer_analysis

# Read updated example code when rerunning this setup cell.
practice_session = reload(customer_analysis).practice_session

## 1. 간단한 Python Method 만들기

간단한 Python 함수를 method로 등록해봅니다.

### 함수 작성과 실행

먼저 DataFrame만 받는 함수를 작성합니다. 

`value` 열의 순위를 계산하고 `group` 열과 함께 반환합니다. 


In [2]:
def rank_values(data, descending=True):
    """Rank observed values, retaining ties."""
    return data[["group", "value"]].assign(
        rank=data["value"].rank(method="min", ascending=not descending).astype(int)
    )


frame = pd.DataFrame({"group": ["A", "B", "C"], "value": [30, 20, 20]})
standalone = rank_values(frame)
standalone

,group,value,rank
0,A,30,1
1,B,20,2
2,C,20,2


### Method로 등록하기

`python_method(...)`는 계산 함수와 다음 정보를 묶어 Method 객체를 만듭니다.

- Method 이름과 버전
- 조회할 데이터의 역할(`DataInput`)
- 필요한 경우 결과 형태(`output`). DataFrame은 자동으로 표로 처리합니다.

`columns`에 필요한 열을 선언합니다. `source="measure"`는 지표, `source="dimension"`은 차원을 받습니다.

**노트북에서는** 
- `practice_session()`으로 테스트 데이터가 준비된 빈 세션을 만들고, 
- `session.register(rank_method)`로 Method를 등록합니다.

관리자 화면과 AI에서 사용하려면 함수와 등록 코드를 프로젝트 파일로 저장해야 합니다. [서비스에 등록하는 방법](../../ko/guides/method-projects.md)을 참고하세요.


In [3]:
rank_method = python_method(
    rank_values,
    name="my_team.rank",
    version="1.0.0",
    inputs={
        "data": DataInput(
            columns={
                "value": Column(source="measure"),
                "group": Column(source="dimension"),
            },
        ),
    },
)

session = await practice_session()
session.register(rank_method)
print("등록된 Method:", [method.name for method in session.list_methods()])

등록된 Method: ['my_team.rank']


### 입력을 선택하고 실행하기

앞에서 만든 Method로 **상품별 매출 순위**를 구하도록 입력을 선택합니다.

- `bindings`: 
  - `value`에 매출(`revenue`)
  - `group`에 상품(`product`)을 연결합니다.
- `params`: `descending=True`로 매출이 큰 상품부터 순위를 매깁니다.
- `scope`: 매출을 조회할 기간과 날짜 기준을 지정합니다.

**노트북 예제에서** 
- 선택한 조건으로 semantic layer에서 상품별 매출을 조회합니다. 
- 조회 결과를 `value`(매출), `group`(상품) 열이 있는 DataFrame으로 만들어 `rank_values` 함수에 전달합니다.

**AI를 통해 “상품별 매출 순위를 알려줘”라고 요청하면,** 
- AI가 등록된 Method를 찾고 semantic catalog에서 매출 지표와 상품 차원을 선택합니다. 
- 매출의 의미나 조회 기간이 모호하면 사용자에게 확인합니다. 
- 입력이 정해지면 데이터를 조회하고 함수를 실행하며, AI는 결과를 읽고 설명합니다.

In [4]:
trial = await session.run(
    "my_team.rank",
    bindings={
        "value": "cube://shop/sales/revenue",
        "group": "cube://shop/sales/product",
    },
    params={"descending": True},
    scope=Scope(
        date_range=("2026-09-01", "2026-09-30"),
        time_dimension="cube://shop/sales/date",
    ),
)

trial.result.primary.data

[{'group': 'A', 'value': 30.0, 'rank': 1},
 {'group': 'B', 'value': 20.0, 'rank': 2},
 {'group': 'C', 'value': 20.0, 'rank': 2}]


## 2. 복잡한 Python Method 만들기
앞선 예제는 Method 등록과 실행 과정을 익히기 위한 것이었습니다.

상품별 순위 정도는 Method를 따로 만들지 않아도 AI가 매번 일관되고 정확하게 만들 수 있습니다.

여러 변수를 다루고 분석 조건을 확인해야 하는 복잡한 코드라면 어떨까요? 

- **데이터 과학자가 검토한 코드를 한 번 등록하고**
- **여러 사용자와 AI가 같은 방법으로 재사용할 수 있다면 편리할 것입니다.** 

Decision Layer는 필요한 데이터를 연결해 이 코드를 실행하고, 입력과 결과를 기록합니다.

이번에는 조금 더 복잡한 OLS 회귀분석을 Method로 만들어 보겠습니다.

### OLS 함수 작성과 등록

statsmodels로 여러 변수를 함께 고려해 관심 있는 변수와 결과의 관계를 계산합니다.

- `data`: 분석할 데이터
- `outcome`: 결과 변수의 열 이름
- `treatment`: 관심 있는 변수의 열 이름
- `confounders`: 함께 고려할 변수들의 열 이름 목록

열 이름을 인자로 받으므로 같은 함수를 다른 데이터에도 사용할 수 있습니다.

이 예제는 수치형 변수만 다룹니다. **결과는 연관성이며, 인과관계를 입증하지는 않습니다.**

이번에는 **고객별 광고 노출 횟수와 구매 금액의 관계**를 분석합니다. 나이와 이전 기간 구매 금액도 함께 고려합니다. 아래 고객 데이터는 예제용 합성 데이터입니다.


In [5]:
def ols(data, outcome, treatment, confounders):
    """다른 변수들을 함께 고려해 관심 변수와 결과의 관계를 OLS로 분석합니다."""
    import numpy as np
    import pandas as pd
    import statsmodels.api as sm

    columns = [outcome, treatment, *confounders]
    if len(set(columns)) != len(columns):
        raise ValueError("같은 열을 두 번 선택했습니다.")

    values = data[columns]
    if not all(pd.api.types.is_numeric_dtype(values[col]) for col in columns):
        raise ValueError("숫자형 열만 선택해 주세요.")
    if not np.isfinite(values.to_numpy(dtype=float)).all():
        raise ValueError("결측값이나 무한대를 먼저 처리해 주세요.")
    if values[outcome].nunique() < 2:
        raise ValueError("결과 변수의 값이 모두 같습니다.")

    design = sm.add_constant(values[[treatment, *confounders]], has_constant="add")
    if len(data) <= design.shape[1]:
        raise ValueError(
            "분석할 변수가 많습니다. 데이터를 늘리거나 변수를 줄여 주세요."
        )
    if np.linalg.matrix_rank(design) < design.shape[1]:
        raise ValueError(
            "설명변수들이 서로 완전히 겹칩니다. 변수 구성을 확인해 주세요."
        )

    fit = sm.OLS(values[outcome], design).fit()
    return {
        "value": float(fit.params[treatment]),
        "n_observations": int(fit.nobs),
        "r_squared": float(fit.rsquared),
        "coefficients": {key: float(value) for key, value in fit.params.items()},
        "causal_claim": "not_established",
        "uncertainty": "not_reported",
    }

In [6]:
from examples.methods.authoring.customer_analysis import customer_sample

customers = customer_sample()

direct_ols = ols(
    customers,
    outcome="purchase_amount",
    treatment="ad_exposures",
    confounders=["age", "prior_purchase_amount"],
)

direct_ols

{'value': 50.00000000000206,
 'n_observations': 200,
 'r_squared': 1.0,
 'coefficients': {'const': 999.999999999618,
  'ad_exposures': 50.00000000000206,
  'age': 9.999999999999813,
  'prior_purchase_amount': 0.20000000000000567},
 'causal_claim': 'not_established',
 'uncertainty': 'not_reported'}

In [7]:
my_ols = python_method(
    ols,
    name="my_team.ols",
    version="1.0.0",
    inputs={
        "data": DataInput(
            columns={
                "entity_id": Column(source="dimension", role="entity_key"),
                "outcome": Column(source="measure", dtype="number"),
                "treatment": Column(
                    source=["dimension", "measure"],
                    dtype="number",
                ),
                "confounders": Column(
                    source=["dimension", "measure"],
                    dtype="number",
                    multiple=True,
                ),
            },
            minimum_rows=10,
        )
    },
    output="estimate",
    requires_packages={"statsmodels": ">=0.14,<0.16"},
)

session.register(my_ols)
print("등록된 Method:", [method.name for method in session.list_methods()])

등록된 Method: ['my_team.rank', 'my_team.ols']


In [8]:
# 고객별 광고 노출 횟수와 구매 금액의 관계를 나이와 이전 구매 금액을 함께 고려해 분석합니다.
ols_trial = await session.run(
    "my_team.ols",
    bindings={
        "entity_id": "cube://shop/customers/id",
        "outcome": "cube://shop/customers/purchase_amount",
        "treatment": "cube://shop/customers/ad_exposures",
        "confounders": [
            "cube://shop/customers/age",
            "cube://shop/customers/prior_purchase_amount",
        ],
    },
    scope=Scope(
        date_range=("2026-02-01", "2026-02-28"),
        time_dimension="cube://shop/customers/date",
    ),
)

ols_trial.result.primary.data

{'value': 50.00000000000206,
 'n_observations': 200,
 'r_squared': 1.0,
 'coefficients': {'const': 999.999999999618,
  'treatment': 50.00000000000206,
  'confounders_0': 9.999999999999813,
  'confounders_1': 0.20000000000000567},
 'causal_claim': 'not_established',
 'uncertainty': 'not_reported'}

### 선택한 입력이 함수에 전달되기까지

**1. 입력을 확인합니다.**

`bindings`에 연결한 지표와 차원이 Cube에 정의되어 있고, Method의 입력 조건에 맞는지 확인합니다.

| 입력 | 연결한 데이터 |
|---|---|
| `entity_id` | 고객 ID |
| `outcome` | 구매 금액 |
| `treatment` | 광고 노출 횟수 |
| `confounders` | 나이, 이전 구매 금액 |

**2. Cube에서 고객별 데이터를 조회합니다.**

선택한 지표와 차원, 기간을 Cube 쿼리로 바꿉니다. Cube는 모델에 정의된 집계와 조인에 따라 고객별 데이터를 반환합니다.

<details>
<summary>Cube 조회 요청 보기</summary>

`/cubejs-api/v1/load`로 보내는 JSON입니다.

```json
{
  "query": {
    "dimensions": ["customers.id", "customers.age"],
    "measures": [
      "customers.purchase_amount",
      "customers.ad_exposures",
      "customers.prior_purchase_amount"
    ],
    "timeDimensions": [
      {
        "dimension": "customers.date",
        "dateRange": ["2026-02-01", "2026-02-28"]
      }
    ],
    "order": [["customers.id", "asc"]],
    "limit": 50000,
    "offset": 0
  }
}
```

</details>

**3. 열 이름을 바꿔 `ols`에 전달합니다.**

구매 금액은 `outcome`, 광고 노출 횟수는 `treatment`가 됩니다. 나이와 이전 구매 금액은 선택한 순서대로 `confounders_0`, `confounders_1`에 담깁니다.

| entity_id | outcome | treatment | confounders_0 | confounders_1 |
|---|---:|---:|---:|---:|
| C001 | 8340 | 10 | 40 | 32200 |
| C002 | 10110 | 6 | 69 | 40600 |

Decision Layer가 이 표와 열 이름으로 함수를 호출합니다.

```python
ols(
    data=조회한_DataFrame,
    outcome="outcome",
    treatment="treatment",
    confounders=["confounders_0", "confounders_1"],
)
```


## 다음 단계

여기까지는 준비된 데이터로 함수를 만들고 로컬에서 실행했습니다.

관리자 화면과 AI에서도 사용하려면 함수와 등록 코드를 프로젝트의 `method.py`에 저장하고 서버를 실행합니다.

[만든 Method를 서비스에서 사용하기](../../ko/guides/method-projects.md)에서 프로젝트 저장, 관리자 화면 테스트, MCP 연결, 실제 Cube 실행 방법을 이어서 확인하세요.
